# Tutorial: Training and Evaluating an MTP Model with MLIPKit

This tutorial demonstrates how to train a Moment Tensor Potential (MTP) model using the `mlipkit` package, evaluate its performance, and visualize the results. We will cover:
- Loading a dataset of atomic structures
- Configuring hyperparameters for MTP training
- Training the model with a subset of the data
- Saving and reloading the trained model
- Evaluating on training and test sets
- Plotting energy, force, and stress comparisons

**Prerequisites:**
- `mlipkit` installed
- MLIP-2 binary (`mlp`) built and accessible
- LAMMPS with MLIP-2 interface (`lmp_mpi`) built
- Training data file: `data/Dataset.traj` (ASE trajectory format)
- Directory for untrained potentials: `data/untrained_pots/`

## 1. Import Required Modules

In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
from ase.io import read
from mlipkit.MTP.mtp_model import MTPModel
from mlipkit.mlip_models import MlipModel

## 2. Load the Dataset

The training data is stored in an ASE trajectory file. We load all configurations.

In [ ]:
dataset_path = 'data/Dataset.traj'
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

dataset = read(dataset_path, index=':')
print(f"Loaded {len(dataset)} configurations from {dataset_path}")

## 3. Review Available MTP Parameters

The `MTPModel.load_doc()` method prints detailed documentation of all configurable parameters.

In [ ]:
MTPModel.load_doc()

## 4. Configure and Create the MTP Model

We define hyperparameters and create an `MTPModel` instance. Only the first 50 structures are used for training to keep the tutorial fast.

In [ ]:
# Paths to binaries (adjust to your system)
mlip2_bin = 'path/to/mlp'
if not os.path.exists(mlip2_bin):
    print(f"Warning: MLIP binary not found at {mlip2_bin}. Training will fail.")

# Hyperparameters for MTP training
hyperparameters = {
    'mlip_bin': mlip2_bin,
    'untrained_pot_file_dir': 'data/untrained_pots/',
    'mtp_level': 6,               # Level of moment tensor expansion
    'min_dist': 'find',           # Automatically determine minimum distance
    'max_dist': 5.5,              # Cutoff radius (Angstroms)
    'radial_basis_type': 'RBChebyshev',
    'radial_basis_size': 6,
    'ene_weight': 1.0,            # Weight for energy error
    'for_weight': 1.0,            # Weight for force error
    'str_weight': 1.0,            # Weight for stress error
    'max_iter': 100,              # Maximum optimization iterations
    'weighting': 'structures',    # Weighting scheme
    'up_mindist': True,           # Update minimum distance during training
    'bin_pref': 'mpirun'          # Parallel execution prefix
}

# Use first 50 structures for training (small subset for demonstration)
training_set = dataset[:50]

# Create model object (not yet saved to disk)
mlip = MTPModel(
    root_dir='MTP',
    name='MTP_model',
    hyperparameters=hyperparameters,
    train_set=training_set
)
print("MTPModel object created. Ready for training.")

## 5. Train the Model

The `train_model()` method runs the MLIP-2 training process. This may take a few minutes depending on the dataset size and hyperparameters.

In [ ]:
mlip.train_model()
print("Training completed. Model saved as JSON in MTP/MTP_model.json")

## 6. Save and Reload the Model

After training, the model is automatically saved. We can reload it from the JSON file using the generic `MlipModel` interface.

In [ ]:
# Delete the original instance to simulate a fresh session
del mlip

# Reload from the saved JSON file
model_path = 'MTP/MTP_model.json'
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file {model_path} not found. Did training succeed?")

mlip = MlipModel.load_model(model_path)
print(f"Model reloaded from {model_path}")

## 7. Evaluate on Training and Test Sets

We evaluate the model's predictions for energies, forces, and stresses. The evaluation uses LAMMPS with the MLIP-2 interface. We'll evaluate on the same 50 structures used for training and on a separate test set (structures 50–100).

In [ ]:
# Path to LAMMPS binary with MLIP-2 interface
lammps_mlip_2_bin = 'path/to/lmp_mpi'
if not os.path.exists(lammps_mlip_2_bin):
    print(f"Warning: LAMMPS binary not found at {lammps_mlip_2_bin}")

compute_params = {
    'lammps_bin': lammps_mlip_2_bin,
    'bin_pref': 'mpirun'
}

# Evaluate on training set
print("Evaluating on training set...")
mlip.evaluate_on_dataset(
    dataset=training_set,
    wdir='train_evaluation',
    parameters=compute_params
)

# Evaluate on test set (next 50 structures)
test_set = dataset[50:100]
print("Evaluating on test set...")
mlip.evaluate_on_dataset(
    dataset=test_set,
    wdir='test_evaluation',
    parameters=compute_params
)
print("Evaluation complete.")

## 8. Visualize Results

The evaluation generates comparison files (`Energy_comparison.dat`, `Forces_comparison.dat`, `Stress_comparison.dat`) in each working directory. We load and plot these to assess model accuracy.

In [ ]:
# Helper function to load comparison data
def load_comparison(filepath):
    """Load two-column comparison file, skipping header."""
    data = np.loadtxt(filepath, skiprows=2).T
    return data[0], data[1]  # reference, predicted

# Training set results
train_dir = 'train_evaluation'
e_train, m_train = load_comparison(os.path.join(train_dir, 'Energy_comparison.dat'))
f_train, f_ml_train = load_comparison(os.path.join(train_dir, 'Forces_comparison.dat'))
s_train, s_ml_train = load_comparison(os.path.join(train_dir, 'Stress_comparison.dat'))

# Test set results
test_dir = 'test_evaluation'
e_test, m_test = load_comparison(os.path.join(test_dir, 'Energy_comparison.dat'))
f_test, f_ml_test = load_comparison(os.path.join(test_dir, 'Forces_comparison.dat'))
s_test, s_ml_test = load_comparison(os.path.join(test_dir, 'Stress_comparison.dat'))

In [ ]:
# Plot training set predictions
fig_train, axes_train = plt.subplots(1, 3, figsize=(15, 5))
fig_train.suptitle('Training Set Performance', fontsize=14)

axes_train[0].plot(e_train, m_train, 'o', markersize=3, alpha=0.7)
axes_train[0].set_xlabel('DFT Energy (eV/atom)')
axes_train[0].set_ylabel('ML Energy (eV/atom)')
axes_train[0].set_title('Energy')

axes_train[1].plot(f_train, f_ml_train, 'o', markersize=3, alpha=0.7)
axes_train[1].set_xlabel('DFT Force (eV/Å)')
axes_train[1].set_ylabel('ML Force (eV/Å)')
axes_train[1].set_title('Forces')

axes_train[2].plot(s_train, s_ml_train, 'o', markersize=3, alpha=0.7)
axes_train[2].set_xlabel('DFT Stress (eV/Å³)')
axes_train[2].set_ylabel('ML Stress (eV/Å³)')
axes_train[2].set_title('Stress')

plt.tight_layout()
plt.show()

In [ ]:
# Plot test set predictions
fig_test, axes_test = plt.subplots(1, 3, figsize=(15, 5))
fig_test.suptitle('Test Set Performance', fontsize=14)

axes_test[0].plot(e_test, m_test, 'o', markersize=3, alpha=0.7)
axes_test[0].set_xlabel('DFT Energy (eV/atom)')
axes_test[0].set_ylabel('ML Energy (eV/atom)')
axes_test[0].set_title('Energy')

axes_test[1].plot(f_test, f_ml_test, 'o', markersize=3, alpha=0.7)
axes_test[1].set_xlabel('DFT Force (eV/Å)')
axes_test[1].set_ylabel('ML Force (eV/Å)')
axes_test[1].set_title('Forces')

axes_test[2].plot(s_test, s_ml_test, 'o', markersize=3, alpha=0.7)
axes_test[2].set_xlabel('DFT Stress (eV/Å³)')
axes_test[2].set_ylabel('ML Stress (eV/Å³)')
axes_test[2].set_title('Stress')

plt.tight_layout()
plt.show()

## 9. Discussion

The plots above show the correlation between DFT reference values and ML predictions. 

**Important notes:**
- The hyperparameters used here (`mtp_level=6`, small radial basis, only 50 training structures) are intentionally minimal for fast demonstration. 
- **Expect poor accuracy** on the test set due to the limited training data and low MTP level.
- For production use, increase `mtp_level` (e.g., 12–16), use a larger training set (hundreds to thousands of structures), and consider optimizing weights and cutoff radius.
- The evaluation step uses LAMMPS; ensure the binary is compiled with MLIP-2 support.

## 10. Next Steps

- Adjust hyperparameters and retrain to improve accuracy.
- Use cross-validation to find optimal settings.
- Deploy the final `.mtp` potential for molecular dynamics simulations.

---
*End of tutorial*